# Notebook 01: Thu thập và Khám phá Dữ liệu (Data Collection & Exploration)
**Đề tài:** Phân tích Sự cố Bất lợi Thuốc Thú y (OpenFDA Animal & Veterinary Adverse Events) giai đoạn 2020–2025 và Dự báo Mức độ Nghiêm trọng năm 2026 (Q1)

**Môn học:** CSC17104 – Lập trình cho Khoa học Dữ liệu  
**Nhóm:** Prg4DS-G3  
- Đinh Công Khang (24120332)
- Trần Ngô Uyên Nhi (24120210)
- Huỳnh Tuấn Kiệt (24120356)

In [ ]:
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Thêm thư mục gốc vào sys.path để import các module trong src/
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.visualization import set_project_plot_style
set_project_plot_style()

---
## 1. Thu thập Dữ liệu (Data Collection — Mục 3.1)

### 1.1. Chủ đề & Bối cảnh thực tế (Subject & Real-world Context)
- **Chủ đề:** Báo cáo các biến cố/phản ứng bất lợi (Adverse Events - AE), thiếu hụt hiệu quả mong đợi (Lack of Expected Effectiveness) và lỗi sản phẩm liên quan đến thuốc/chế phẩm sinh học dùng trong thú y.
- **Bối cảnh thực tế:** Giám sát cảnh giác dược thú y (Veterinary Pharmacovigilance) giúp cơ quan quản lý (FDA), bác sĩ thú y và người nuôi thú cưng/gia súc nhận diện sớm các rủi ro an toàn thuốc theo loài, độ tuổi, đường dùng và hoạt chất.

### 1.2. Nguồn dữ liệu & Phạm vi thời gian (Source & Temporal Scope)
- **Tổ chức phát hành:** U.S. Food and Drug Administration (FDA) – Center for Veterinary Medicine (CVM).
- **Cổng tải dữ liệu:** [openFDA Animal & Veterinary Adverse Events Downloads](https://open.fda.gov/data/downloads/)
- **Phạm vi thời gian sử dụng trong đồ án:**
  - **Tập lịch sử & huấn luyện (2020 – 2025):** Toàn bộ các phân vùng theo quý (Q1–Q4 của 6 năm 2020, 2021, 2022, 2023, 2024, 2025).
  - **Tập kiểm thử tương lai / Out-of-Time Test (2026 Q1):** Dữ liệu quý đầu năm 2026 (do năm 2026 hiện mới công bố Q1, chưa có Q2, Q3 và Q4), dùng để đối chiếu xu hướng và đánh giá độ chính xác thực tế của mô hình học máy.

### 1.3. Giấy phép & Điều khoản sử dụng (License & Terms)
- **License:** Creative Commons CC0 1.0 Universal (Public Domain Dedication) — [https://open.fda.gov/license/](https://open.fda.gov/license/)
- **Terms of Service:** [https://open.fda.gov/terms/](https://open.fda.gov/terms/)

### 1.4. Phương pháp thu thập & Hạn chế/Thiên lệch đã biết (Collection Method & Known Biases)
- **Phương pháp thu thập:** Hệ thống báo cáo tự nguyện và bắt buộc gửi về FDA CVM từ các nhà sản xuất thuốc, bác sĩ thú y (`Veterinarian`), chủ vật nuôi (`Animal Owner`) và nhân viên y tế khác.
- **Hạn chế/Thiên lệch (Limitations & Biases):**
  1. *Under-reporting & Reporting bias:* Không phải mọi ca phản ứng phụ ngoài thực tế đều được báo cáo; các ca nặng (`serious_ae = true`) hoặc thuốc mới thường được chú ý báo cáo nhiều hơn.
  2. *Không có mẫu số phơi nhiễm (No denominator):* Dữ liệu chỉ gồm các ca có sự cố, không có tổng số cá thể động vật đã dùng thuốc nhưng bình thường $\rightarrow$ chỉ kết luận về mối liên hệ/tỷ trọng trong tập báo cáo, không suy diễn thành nguy cơ tuyệt đối hay quan hệ nhân quả.
  3. *Dữ liệu lồng nhau & không đồng nhất:* Định dạng JSON đa cấp (`drug[]`, `reaction[]`, `outcome[]`), đơn vị tuổi/cân nặng đa dạng cần chuẩn hóa kỹ.

### 1.5. Lý do chọn bộ dữ liệu (Motivation)
- Độc đáo, mang tính ứng dụng liên ngành (Y Dược Thú y & Khoa học Dữ liệu), quy mô lớn (hàng trăm nghìn báo cáo qua giai đoạn 2020–2026 Q1) và cấu trúc JSON lồng nhau giúp thể hiện trọn vẹn kỹ năng làm phẳng (flatten/normalize), tiền xử lý và thiết kế kiểm thử theo thời gian (Temporal Validation).

In [ ]:
# TODO [Giai đoạn 1]: Đọc các file JSON (2020 Q1 - 2026 Q1) từ thư mục ../data/raw/
# và tách thành 4 bảng chuẩn hóa: reports_df, drugs_df, reactions_df, outcomes_df


---
## 2. Khám phá Dữ liệu (Data Exploration — Mục 3.2)

### 2.1. Tổng quan cấu trúc & Schema (Overview)
- Kiểm tra số dòng, số cột của từng bảng sau khi flatten (`reports`, `drugs`, `reactions`, `outcomes`) qua từng năm (2020–2025 vs. 2026 Q1).
- Ý nghĩa một dòng ở mỗi bảng và khóa liên kết `unique_aer_id_number` / `report_id`.
- Kiểm tra các bản ghi trùng lặp (`duplicated`) hoặc rỗng và nêu rõ lý do giữ/loại bỏ.

In [ ]:
# TODO [Giai đoạn 2]: Thống kê kích thước bảng, kiểu dữ liệu và kiểm tra duplicate theo unique_aer_id_number


### 2.2. Khám phá các biến số (Numerical Columns)
- Phân phối, trung tâm và độ phân tán của: tuổi vật nuôi (`animal.age`), cân nặng (`animal.weight`), số lượng cá thể bị ảnh hưởng (`number_of_animals_affected`), độ trễ báo cáo (`reporting_delay_days`), số thuốc dùng kèm (`num_drugs`).
- Phát hiện giá trị ngoại lai (IQR, Z-score) và các giá trị vô lý/placeholder (tuổi âm, cân nặng = 0 hoặc cực đại bất thường) kèm hướng xử lý.

In [ ]:
# TODO [Giai đoạn 2]: Phân tích thống kê mô tả và trực quan hóa phân phối biến số (sau khi quy đổi đơn vị)


### 2.3. Khám phá các biến phân loại (Categorical Columns)
- Kiểm tra số lượng giá trị phân biệt và tần suất của: `animal.species`, `animal.gender`, `primary_reporter`, `type_of_information`, `serious_ae`, `treated_for_ae`, `drug.route`, `drug.used_according_to_label`, `reaction.veddra_term_name`, `outcome.medical_status`.
- Phát hiện nhãn không nhất quán (in hoa/thường, từ đồng nghĩa) và gom nhóm các danh mục hiếm (`Other`).

In [ ]:
# TODO [Giai đoạn 2]: Thống kê tần suất biến phân loại và kiểm tra nhãn không đồng nhất


### 2.4. Phân tích Dữ liệu Khuyết (Missing Data Analysis)
- Số lượng và tỷ lệ % giá trị khuyết trên từng cột.
- Đánh giá cơ chế khuyết (MCAR / MAR / MNAR): ví dụ `onset_date`, `weight`, hoặc `serious_ae` bị khuyết có phụ thuộc vào `primary_reporter` hay `type_of_information` không?
- Chiến lược xử lý cho từng cột (Drop, Impute hay tách thành nhóm `'Unknown'` riêng).

In [ ]:
# TODO [Giai đoạn 2]: Bảng tổng hợp tỷ lệ missing và biểu đồ trực quan hóa mẫu khuyết


### 2.5. Mối quan hệ sơ bộ giữa các biến (Relationships)
- Tương quan giữa các biến số và bảng chéo (cross-tabulations) giữa các tổ hợp quan trọng (`species` $\times$ `serious_ae`, `type_of_information` $\times$ `primary_reporter`, xu hướng số lượng báo cáo theo quý từ 2020Q1 đến 2026Q1).

In [ ]:
# TODO [Giai đoạn 2]: Ma trận tương quan và cross-tabulation sơ bộ


### 2.6. Tổng kết Khám phá Dữ liệu & Kế hoạch Tiền xử lý (EDA Summary & Preprocessing Plan)
1. **3–5 Quan sát cốt lõi (Key Observations):** *(Cập nhật sau khi chạy dữ liệu)*
2. **Các vấn đề chất lượng dữ liệu nghiêm trọng (Serious Data Quality Issues):** *(Đơn vị tuổi/cân nặng hỗn hợp, cấu trúc 1-nhiều lồng nhau, tỷ lệ khuyết ở các trường tùy chọn...)*
3. **Kế hoạch Tiền xử lý chuẩn hóa (Preprocessing Plan):** Lưu các bảng đã làm sạch vào `data/processed/` để sử dụng thống nhất cho Notebook 02 và Notebook 03.

In [ ]:
# TODO [Giai đoạn 2]: Lưu dữ liệu đã chuẩn hóa cơ bản vào ../data/processed/
